# IL ion pKa by DFT + SMD (extended run, 453 ions)

**How to run:** Runtime → Change runtime type → **T4 GPU** → run the cells top to bottom. When Step 2 asks, upload `IL_ions_for_DFT_input.xlsx`.

- Results are saved to Drive (`MyDrive/IL_pKa_DFT/energies.jsonl`) after every molecule. If Colab disconnects, reconnect and rerun the same cell: it resumes and skips finished molecules.
- If the old `energies.jsonl` is still in that Drive folder, the 420-ion results are reused and only the new ions are computed.
- Keep the settings fixed (B3LYP/def2-SVPD, SMD water, GPU). Do not mix with CPU results.
- Final file: `IL_pKa_DFT_SMD.csv` in the same Drive folder. Send it back to be merged.

**If Colab crashes with 'out of RAM':** Runtime → Restart session, then run all cells from the top again. Finished molecules are on Drive and are skipped. This version also frees memory after each molecule and stops cleanly if RAM passes 85%.


# Physics-based pKa for the 420 IL ions — DFT + SMD on a free Colab GPU

**Non-ML method.** For each ion we compute the energy of both members of its acid–base pair in water
with DFT and the SMD solvent model, then convert the energy difference to a pKa with a straight-line
calibration fitted on known values — the same idea as Klamt *et al.* (2003) for COSMO-RS.

$$\Delta E = E(\text{base form}) - E(\text{acid form}) \qquad\Rightarrow\qquad pK_a = a\,\Delta E + b$$

- **anion A⁻** → acid form **HA** (exactly one extra H⁺) → gives the pKa of the conjugate acid
- **basic cation BH⁺** → base form **B** (exactly one H⁺ removed) → gives pKaH
- **permanent cations** → no pair exists → N/A

## Why this, and not the ML models

Both ML models failed badly on strong acids (sulfonates predicted near 9, true value about −2), because their
training data only covered pKa 2–12. This method has no training range: the calibration itself includes your
strong acids.

## What a CPU pilot already showed (8 acids, same method)

| Acid | True pKa | Leave-one-out prediction |
|---|---|---|
| acetate | 4.76 | 5.12 |
| benzoate | 4.20 | 3.94 |
| formate | 3.75 | 2.90 |
| methanesulfonate | −1.92 | −1.26 |
| trifluoroacetate | 0.23 | −3.87 |
| chloride | −6.30 | −1.90 |

The fitted slope came out at **0.43 pKa per kcal/mol** — almost exactly what Klamt's COSMO-RS paper found
(58% of the theoretical 0.733). Carboxylic acids were good; the strong acids landed in the right region
(strongly acidic) but not precisely. Overall leave-one-out error was 1.7 pKa units on 8 acids — small sample,
which is what the full calibration below is for.

## Free Colab reality

- You get a **T4**, which is slow at the double-precision maths DFT needs. It works, just not fast.
- Sessions disconnect. **Every result is saved to Google Drive as soon as it finishes**; rerunning any
  cell skips finished molecules and carries on.
- Each run stops itself after `RUN_MINUTES` so Colab doesn't kill it mid-molecule.
- **Run order:** Steps 1–4 once. Then Step 5 (pilot). Then Steps 6 and 8 as many sessions as needed.

---

## Step 1 — Install

Runtime → Change runtime type → **T4 GPU**. Then run this cell. If the GPU package fails to install or import,
the notebook falls back to CPU automatically (correct, but much slower).

In [ ]:
!pip install -q pyscf gpu4pyscf-cuda12x cutensor-cu12 rdkit openpyxl

import os, json, time, math
import numpy as np, pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')
import pyscf
from pyscf import gto

USE_GPU = False
try:
    import cupy
    from gpu4pyscf import dft as gdft
    _t = pyscf.M(atom='O 0 0 0.117; H -0.757 0 -0.470; H 0.757 0 -0.470', basis='sto-3g', verbose=0)
    _m = gdft.rks.RKS(_t, xc='HYB_GGA_XC_B3LYP').density_fit().SMD()
    _m.with_solvent.solvent = 'water'
    _m.kernel()
    USE_GPU = True
except Exception as e:
    print('GPU path unavailable -> using CPU. Reason:', repr(e)[:200])
print('pyscf', pyscf.__version__, '| USE_GPU =', USE_GPU)

## Step 2 — Settings and Google Drive

All results go to one folder on your Drive. **Keep these settings fixed for the whole project** — mixing
methods, basis sets or CPU/GPU results would make the energies incomparable. The code enforces this.

In [ ]:
XC        = 'HYB_GGA_XC_B3LYP'   # functional
BASIS     = 'def2-svpd'          # has diffuse functions (needed for anions) and covers iodine
SOLVENT   = 'water'
N_CONF    = 10                   # RDKit conformers tried per molecule; lowest MMFF energy kept
MAX_HEAVY = 40                   # skip very large ions (flagged, not silently dropped)
RUN_MINUTES = 240                # each run stops itself after this (4 h); rerun the cell to continue

RESULTS_DEVICE = 'gpu'   # your saved results were computed on GPU; keep this as 'gpu'
ALLOW_CPU = False        # set automatically by the CPU/GPU check in Step 5b -- don't edit by hand
LEVEL = f"{XC}/{BASIS}+ecp/SMD-{SOLVENT}/{RESULTS_DEVICE}"

try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/IL_pKa_DFT'
except ImportError:
    OUT = './IL_pKa_DFT'
os.makedirs(OUT, exist_ok=True)
CACHE = f'{OUT}/energies.jsonl'
print('Results folder:', OUT, '\nLevel of theory:', LEVEL)

## Step 3 — Acid–base pairs that differ by exactly one proton

This matters more than it looks. "Neutralize the ion" is wrong for multiply charged anions: sulfate SO₄²⁻
would become H₂SO₄, but its conjugate acid is HSO₄⁻. Nine of your anions carry −2 or −3 (sulfate, malonate,
succinate, malate, aspartate, tartrate, glutamate, citrate, phosphonate), so this uses one-proton pairs
throughout.

In [ ]:
# aliphatic amine N: not amide / sulfonamide, not on an aromatic ring, not charged
BASIC_N = Chem.MolFromSmarts('[NX3;+0;!$(N-[C,S,P]=[O,S,N]);!$(N-a);!$(N-[C,S]#*)]')

def add_one_proton(smiles):
    """Anion -> conjugate acid: exactly one extra H+.
    If the anion has a basic aliphatic amine (glycinate, taurinate, aspartate...), the proton goes on that N,
    giving the zwitterion -- that's the real conjugate acid (pKa ~9-10, the IUPAC and sheet value).
    Otherwise it goes on a negative atom (O preferred)."""
    mol = Chem.MolFromSmiles(smiles)
    neg = [a for a in mol.GetAtoms() if a.GetFormalCharge() < 0]
    if not neg: return None
    amine = mol.GetSubstructMatches(BASIC_N)
    if amine:
        a = mol.GetAtomWithIdx(amine[0][0])
        a.SetFormalCharge(1); a.SetNumExplicitHs(a.GetTotalNumHs() + 1); a.SetNoImplicit(True)
        try:
            Chem.SanitizeMol(mol); return Chem.MolToSmiles(mol)
        except Exception:
            return None
    order = {'O': 0, 'N': 1, 'S': 2, 'C': 3}
    neg.sort(key=lambda a: order.get(a.GetSymbol(), 9))
    a = neg[0]
    single_halide = mol.GetNumHeavyAtoms() == 1 and a.GetSymbol() in ('F', 'Cl', 'Br', 'I')
    if a.GetSymbol() not in order and not single_halide:
        return None                                  # e.g. BF4-, SbF6-, I3-
    a.SetFormalCharge(a.GetFormalCharge() + 1)
    a.SetNumExplicitHs(a.GetTotalNumHs() + 1); a.SetNoImplicit(True)
    try:
        Chem.SanitizeMol(mol); return Chem.MolToSmiles(mol)
    except Exception:
        return None

COOH = Chem.MolFromSmarts('[CX3](=O)[OX2H1]')

def remove_one_proton(smiles):
    """Protonated cation BH+ -> conjugate base: remove exactly one H+.
    Amino-acid-type cations (aliphatic NH3+ plus a COOH): the COOH proton (pKa ~2) leaves first,
    giving the zwitterion -- that's the IUPAC pKaH1 value. Otherwise remove it from the positive atom."""
    mol = Chem.MolFromSmiles(smiles)
    pos = [a for a in mol.GetAtoms() if a.GetFormalCharge() > 0 and a.GetTotalNumHs() > 0]
    if not pos: return None                          # permanent cation
    acid = mol.GetSubstructMatches(COOH)
    if acid and pos[0].GetSymbol() == 'N' and not pos[0].GetIsAromatic():
        o = mol.GetAtomWithIdx(acid[0][2])
        o.SetFormalCharge(-1); o.SetNumExplicitHs(0); o.SetNoImplicit(True)
        try:
            Chem.SanitizeMol(mol); return Chem.MolToSmiles(mol)
        except Exception:
            return None
    a = pos[0]
    a.SetFormalCharge(a.GetFormalCharge() - 1)
    a.SetNumExplicitHs(a.GetTotalNumHs() - 1); a.SetNoImplicit(True)
    try:
        Chem.SanitizeMol(mol); return Chem.MolToSmiles(mol)
    except Exception:
        return None

from rdkit.Chem.MolStandardize import rdMolStandardize
_reion = rdMolStandardize.Reionizer()

def make_pair(smiles, ion_type):
    """The ONE place acid-base pairs are built. Returns (acid_form, base_form, kind).
    - anions: first move the charge to the most acidic site (fixes salicylates drawn as phenoxide)
    - 'kind' = which calibration line, decided by WHERE the proton moves:
        proton on/off an amine N  -> 'base' line  (e.g. glycinate <-> zwitterion is an ammonium reaction)
        proton on/off O, S, halogen -> 'acid' line (e.g. amino-acid cation losing its COOH proton)"""
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        return None, None, None
    if ion_type == 'anion':
        base = Chem.MolToSmiles(_reion.reionize(m))
        acid = add_one_proton(base)
        kind = 'base' if Chem.MolFromSmiles(base).HasSubstructMatch(BASIC_N) else 'acid'
    else:
        acid = smiles
        base = remove_one_proton(smiles)
        pos = [a for a in m.GetAtoms() if a.GetFormalCharge() > 0 and a.GetTotalNumHs() > 0]
        cooh_case = bool(pos) and m.HasSubstructMatch(COOH) and pos[0].GetSymbol() == 'N' and not pos[0].GetIsAromatic()
        kind = 'acid' if cooh_case else 'base'
    return acid, base, kind

def protonate_atom(smiles, idx):
    mol = Chem.MolFromSmiles(smiles); a = mol.GetAtomWithIdx(idx)
    a.SetFormalCharge(a.GetFormalCharge() + 1)
    a.SetNumExplicitHs(a.GetTotalNumHs() + 1); a.SetNoImplicit(True)
    Chem.SanitizeMol(mol); return Chem.MolToSmiles(mol)

def deprotonate_atom(smiles, idx):
    mol = Chem.MolFromSmiles(smiles); a = mol.GetAtomWithIdx(idx)
    a.SetFormalCharge(a.GetFormalCharge() - 1)
    a.SetNumExplicitHs(a.GetTotalNumHs() - 1); a.SetNoImplicit(True)
    Chem.SanitizeMol(mol); return Chem.MolToSmiles(mol)

for s in ['O=S(=O)([O-])[O-]', 'O=S(=O)([O-])O', 'CC(=O)[O-]', '[Cl-]', 'F[B-](F)(F)F', 'NCC(=O)[O-]', 'NCCS(=O)(=O)[O-]']:
    print(f'{s:22s} -> acid form {add_one_proton(s)}')
for s, t in [('Cc1cccc(C(=O)O)c1[O-]','anion'), ('NCC(=O)[O-]','anion'), ('CC(=O)[O-]','anion'),
             ('[NH3+]CC(=O)O','cation'), ('c1cc[nH+]cc1','cation')]:
    print(f'{s:24s} -> acid {str(make_pair(s,t)[0]):26s} base {str(make_pair(s,t)[1]):24s} line: {make_pair(s,t)[2]}')
for s in ['c1cc[nH+]cc1', 'C[N+](C)(C)CCO', '[NH3+]CC(=O)O', 'CSCCC([NH3+])C(=O)O']:
    print(f'{s:22s} -> base form {remove_one_proton(s)}')

## Step 4 — The energy calculation

Geometry: RDKit generates `N_CONF` conformers, force-field optimises them, and the lowest-energy one is used.
Energy: a single-point DFT calculation with SMD water (no DFT geometry optimisation — that would multiply the
cost several times on a T4). Systematic errors from these shortcuts are partly absorbed by the calibration,
because the calibration molecules go through exactly the same procedure.

In [ ]:
def best_conformer_xyz(smiles, n_conf=N_CONF, seed=42):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    if mol.GetNumAtoms() == 1:
        return f"{mol.GetAtomWithIdx(0).GetSymbol()} 0 0 0"
    p = AllChem.ETKDGv3(); p.randomSeed = seed
    cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, params=p))
    if not cids:
        p.useRandomCoords = True
        cids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, params=p))
    if not cids:
        raise RuntimeError('3D embedding failed')
    if AllChem.MMFFHasAllMoleculeParams(mol):
        res = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=2000)
    else:
        res = AllChem.UFFOptimizeMoleculeConfs(mol, maxIters=2000)
    best = cids[int(np.argmin([e for _, e in res]))]
    conf = mol.GetConformer(best)
    return '\n'.join(f'{a.GetSymbol()} {conf.GetAtomPosition(a.GetIdx()).x:.6f} '
                     f'{conf.GetAtomPosition(a.GetIdx()).y:.6f} {conf.GetAtomPosition(a.GetIdx()).z:.6f}'
                     for a in mol.GetAtoms())

import gc, psutil
RAM_LIMIT = 85          # stop cleanly when system RAM use passes this percent
def ram_percent():
    return psutil.virtual_memory().percent
def free_memory():
    gc.collect()
    if USE_GPU:
        try:
            import cupy
            cupy.get_default_memory_pool().free_all_blocks()
            cupy.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass

def smd_energy(smiles):
    charge = Chem.GetFormalCharge(Chem.MolFromSmiles(smiles))
    # ecp=BASIS is essential: def2 basis sets assume an effective core potential for iodine (and heavier).
    # Without it, iodine is computed with all 53 electrons in a basis built for 25 -> garbage energies.
    mol = gto.M(atom=best_conformer_xyz(smiles), charge=charge, spin=0, basis=BASIS, ecp=BASIS, verbose=0)
    if USE_GPU:
        mf = gdft.rks.RKS(mol, xc=XC).density_fit().SMD()
    else:
        from pyscf import dft, solvent
        mf = solvent.SMD(dft.RKS(mol, xc=XC).density_fit())
    mf.with_solvent.solvent = SOLVENT
    mf.max_cycle = 150
    t = time.time()
    e = float(mf.kernel())
    out = dict(E=e, converged=bool(mf.converged), seconds=round(time.time() - t, 1), charge=charge)
    del mf, mol          # release the big arrays before the next molecule
    free_memory()
    return out

def load_cache():
    done = {}
    if os.path.exists(CACHE):
        for line in open(CACHE):
            try:
                r = json.loads(line)
            except Exception:
                continue                              # half-written line from a crash -- ignore it
            if r.get('level') == LEVEL:               # never mix levels of theory
                done[r['smiles']] = r
    return done

def run_species(smiles_list, minutes=RUN_MINUTES):
    """Compute every species not yet on Drive. Saves after each one; stops before the time budget."""
    done = load_cache()
    todo = [s for s in dict.fromkeys(smiles_list) if s and s not in done]
    if todo and RESULTS_DEVICE == 'gpu' and not USE_GPU and not ALLOW_CPU:
        print(f'{len(todo)} molecules still need computing, but this is a CPU runtime.')
        print('Not computing them here (it would mix CPU and GPU results). Switch to a GPU runtime to compute.')
        return False
    todo.sort(key=lambda s: Chem.MolFromSmiles(s).GetNumHeavyAtoms())   # small first
    print(f'{len(done)} already on Drive, {len(todo)} to compute.')
    start = time.time()
    for i, s in enumerate(todo, 1):
        if ram_percent() > RAM_LIMIT:
            print(f'RAM use is {ram_percent():.0f}% -- stopping cleanly after {i-1} molecules, before Colab crashes.')
            print('Fix: Runtime -> Restart session, run the cells from the top again, then rerun this cell. Finished molecules are skipped.')
            return False
        if (time.time() - start) / 60 > minutes:
            print(f'Time budget reached after {i-1} molecules. Rerun this cell to continue.')
            return False
        if Chem.MolFromSmiles(s).GetNumHeavyAtoms() > MAX_HEAVY:
            rec = dict(E=None, converged=False, seconds=0, charge=None, note=f'skipped: >{MAX_HEAVY} heavy atoms')
        else:
            try:
                rec = smd_energy(s)
            except Exception as e:
                rec = dict(E=None, converged=False, seconds=0, charge=None, note=f'error: {repr(e)[:120]}')
        rec.update(smiles=s, level=LEVEL)
        with open(CACHE, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        print(f'[{i}/{len(todo)}] {s[:45]:45s} {rec.get("seconds",0):6.0f}s  conv={rec["converged"]}'
              + (f'  {rec.get("note","")}' if rec.get('note') else ''), flush=True)
    print('All done.')
    return True
print('Energy functions ready.')

## Step 5 — Pilot: check speed before committing hours

Five small acid–base pairs. Look at the seconds per molecule, then the projected total below.
If it projects far beyond what you can spare, raise `MAX_HEAVY` less, or run only the calibration first.

In [ ]:
pilot = ['O=CO', 'O=C[O-]', 'CC(=O)O', 'CC(=O)[O-]', 'CS(=O)(=O)O', 'CS(=O)(=O)[O-]',
         'Cl', '[Cl-]', 'c1cc[nH+]cc1', 'c1ccncc1']
run_species(pilot, minutes=30)
d = load_cache()
secs = [d[s]['seconds'] for s in pilot if s in d and d[s]['converged']]
heavy = [Chem.MolFromSmiles(s).GetNumHeavyAtoms() for s in pilot if s in d and d[s]['converged']]
print(f'\nAverage {np.mean(secs):.0f}s per molecule for {np.mean(heavy):.1f} heavy atoms on average.')
print('Larger ions cost roughly (heavy atoms)^3 more -- the jobs in Step 6 print their own projection.')

## Step 5b — Only if you're on a CPU runtime: is CPU safe to use?

Your saved results came from the GPU. This recomputes 3 of them on the CPU and compares. If they agree to
within 0.1 kcal/mol (about 0.04 pKa units — negligible), CPU results can safely go in the same file, and the
calculation cells will run on CPU. If not, they stay blocked and you'll need a GPU for new calculations.
Takes about 2 minutes. Skip this on a GPU runtime.

In [ ]:
d = load_cache()
test = [s for s in ['O=CO', 'O=C[O-]', 'CC(=O)O', 'CC(=O)[O-]', 'c1ccncc1'] if s in d][:3]
if USE_GPU:
    print('GPU runtime -- no check needed.')
elif len(test) < 3:
    print('Not enough existing GPU results to compare against -- CPU stays blocked.')
else:
    diffs = []
    for s in test:
        e_cpu = smd_energy(s)['E']
        diff = (e_cpu - d[s]['E']) * 627.509
        diffs.append(abs(diff))
        print(f"{s:14s} GPU {d[s]['E']:.6f}  CPU {e_cpu:.6f}  difference {diff:+.4f} kcal/mol")
    ALLOW_CPU = max(diffs) < 0.1
    print('\nSAFE: CPU calculations allowed.' if ALLOW_CPU else '\nNOT SAFE: CPU and GPU disagree -- CPU stays blocked.')

## Step 6 — Build the calibration set and run it

**Calibration molecules (all with sourced experimental values):**

1. Your sheet's 35 literature values — minus the 3 amino-acid anions (their value is a different equilibrium),
   and with the three sheet corrections applied (hydrogencarbonate 6.35, dihydrogen-phosphate 2.15,
   hydrogensulfate −3, from a university textbook acidity table).
2. **Measured values from the IUPAC Digitized pKa Dataset** for ions that have no sheet value (about 70 of
   your 211 unknowns). These get their measured value directly and also sharpen the calibration.
3. Your sheet has only 2 known basic cations — too few to fit a line. So 20 small bases and 15 small acids are
   added from the Baltruschat & Czodrowski (2020) experimental dataset (CC BY 4.0), picked automatically:
   one ionizable site only (so there's no doubt which atom the proton goes on), at most 10 heavy atoms,
   spread across the pKa range, fixed random seed.

**Upload your Excel file when prompted** (or put it in the Drive folder from Step 2).

In [ ]:
xlsx = f'{OUT}/IL_ions_for_DFT_input.xlsx'
if not os.path.exists(xlsx):
    from google.colab import files
    up = files.upload(); name = list(up.keys())[0]
    os.replace(name, xlsx)
ions = pd.read_excel(xlsx, sheet_name='All_ions')

# The sheet used the anion's OWN pKa2 for these amphiprotic 'hydrogen-X' anions; its rule needs the
# conjugate acid's pKa1. Values: textbook table (first 3) and the IUPAC Digitized pKa Dataset (rest).
CORRECTIONS = {'hydrogencarbonate': 6.35, 'dihydrogen-phosphate': 2.15, 'hydrogensulfate': -3.0,
               'AN_020': 1.20, 'AN_028': 2.85, 'AN_033': 3.36, 'AN_038': 2.74, 'AN_032': 3.10, 'AN_034': 4.30}

cal = []   # rows: id, kind, acid_form, base_form, pKa, source
for _, r in ions[ions['status'] == 'Source-backed value'].iterrows():
    pka = CORRECTIONS.get(r['master_id'], CORRECTIONS.get(r['name'], float(r['aqueous_pKa'])))
    acid, base, kind = make_pair(r['smiles'], r['ion_type'])
    if acid and base:
        cal.append(dict(id=r['master_id'], kind=kind, acid_form=acid, base_form=base, pKa=pka, source='sheet'))

# Measured values from the IUPAC Digitized pKa Dataset (critically assessed, CC BY-NC 4.0).
# Any "unknown" ion found here gets its MEASURED value, and becomes an extra calibration point.
!git clone -q https://github.com/IUPAC/Dissociation-Constants.git 2>/dev/null || true
from rdkit.Chem.MolStandardize import rdMolStandardize
_unch = rdMolStandardize.Uncharger()
def parent_key(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return None
    m = _unch.uncharge(m); Chem.RemoveStereochemistry(m); return Chem.MolToSmiles(m)
iu = pd.read_csv('Dissociation-Constants/iupac_high-confidence_v2_4.csv', low_memory=False)
iu['val'] = pd.to_numeric(iu.pka_value, errors='coerce'); iu['Tn'] = pd.to_numeric(iu['T'], errors='coerce')
iu = iu[iu.val.notna() & iu.cosolvent.isna() & iu.Tn.between(18, 30)]     # water only, near room T
iu['key'] = iu.SMILES.map(parent_key)
IU = iu.groupby(['key', 'pka_type']).val.median()                           # median of repeat measurements
def iupac_value(smiles, ion_type):
    q = Chem.GetFormalCharge(Chem.MolFromSmiles(smiles))
    # -1 anion -> pKa1 of neutral parent, -2 -> pKa2 ... ; +1 cation -> pKaH1
    t = f'pKa{-q}' if (ion_type == 'anion' and q < 0) else ('pKaH1' if (ion_type == 'cation' and q == 1) else None)
    return IU.get((parent_key(smiles), t), np.nan) if t else np.nan
IUPAC_VALUES = {}
for _, r in ions[ions['status'] != 'Source-backed value'].iterrows():
    v = iupac_value(r['smiles'], r['ion_type'])
    if pd.isna(v):
        continue
    IUPAC_VALUES[r['master_id']] = float(v)
    acid, base, kind = make_pair(r['smiles'], r['ion_type'])
    if acid and base:
        cal.append(dict(id=r['master_id'], kind=kind, acid_form=acid, base_form=base, pKa=float(v), source='IUPAC'))
print(f'{len(IUPAC_VALUES)} ions without a sheet value have MEASURED IUPAC values -> calibration, not prediction')

# extra calibration molecules from the experimental training set
!git clone -q https://github.com/czodrowskilab/Machine-learning-meets-pKa.git 2>/dev/null || true
from rdkit.Chem import PandasTools
ext = PandasTools.LoadSDF('Machine-learning-meets-pKa/datasets/combined_training_datasets_unique_no_oe.sdf')
ext['pKa'] = ext['pKa'].astype(float)

def pick(kind, n, seed=42):
    rows = []
    for mol, pka, t, mid in zip(ext.ROMol, ext.pKa, ext.marvin_pKa_type, ext.ID):
        if t != kind or mol is None or Chem.GetFormalCharge(mol) != 0 or mol.GetNumHeavyAtoms() > 10:
            continue
        smi = Chem.MolToSmiles(mol); m = Chem.MolFromSmiles(smi)
        if kind == 'acidic':
            sites = [a.GetIdx() for a in m.GetAtoms() if a.GetSymbol() != 'C' and a.GetTotalNumHs() >= 1]
        else:
            sites = [a.GetIdx() for a in m.GetAtoms() if a.GetSymbol() == 'N']
        if len(sites) != 1:
            continue
        try:
            if kind == 'acidic':
                rows.append(dict(id=f'ext_{mid}', kind='acid', acid_form=smi,
                                 base_form=deprotonate_atom(smi, sites[0]), pKa=pka, source='Baltruschat2020'))
            else:
                rows.append(dict(id=f'ext_{mid}', kind='base', acid_form=protonate_atom(smi, sites[0]),
                                 base_form=smi, pKa=pka, source='Baltruschat2020'))
        except Exception:
            pass
    df = pd.DataFrame(rows).drop_duplicates('acid_form')
    df['bin'] = pd.cut(df.pKa, bins=5, labels=False)          # spread across the pKa range
    per = max(1, n // df['bin'].nunique())
    return (df.groupby('bin', group_keys=False)
              .apply(lambda g: g.sample(min(per, len(g)), random_state=seed))
              .head(n).drop(columns='bin'))

cal = pd.concat([pd.DataFrame(cal), pick('acidic', 15), pick('basic', 20)], ignore_index=True)
cal.to_csv(f'{OUT}/calibration_set.csv', index=False)
print(cal.groupby(['kind', 'source']).size().to_string())
print(f'Total calibration pairs: {len(cal)} -> {2*len(cal)} DFT calculations')

In [ ]:
# Run the calibration calculations. Rerun this cell after a disconnect -- it resumes.
run_species(list(cal.acid_form) + list(cal.base_form))

## Step 7 — Fit the calibration and measure honest error

Separate straight lines for acids and bases. **Leave-one-out error** is the number to trust: each molecule is
predicted by a line fitted without it. The per-family breakdown shows where the method is reliable.

In [ ]:
H2K = 627.509
d = load_cache()
def dE_for(row):
    a, b = d.get(row['acid_form']), d.get(row['base_form'])
    if not a or not b or not a['converged'] or not b['converged']:
        return np.nan
    return (b['E'] - a['E']) * H2K          # deprotonation energy in solution, kcal/mol

def family(smiles):
    m = Chem.MolFromSmiles(smiles)
    pats = [('sulfonate/sulfate', '[SX4](=O)(=O)[OX1-,OX2H1]'), ('carboxylate', '[CX3](=O)[OX1-,OX2H1]'),
            ('halide', '[F,Cl,Br,I;X0,H1;!$(*~*)]'), ('phenol', '[OX2H1,OX1-]c'), ('amine/N-base', '[#7]')]
    for name, p in pats:
        if m.HasSubstructMatch(Chem.MolFromSmarts(p)):
            return name
    return 'other'

# Left out of the fit, with reasons (they stay in the table, they just don't shape the line):
#  - halides: bare single-atom ions are SMD's known weak spot, and all have literature values anyway
#  - > CAL_MAX_HEAVY heavy atoms: one force-field conformer is not reliable for large, floppy molecules
CAL_MAX_HEAVY = 20
cal['heavy'] = [Chem.MolFromSmiles(x).GetNumHeavyAtoms() for x in cal.base_form]
cal['dE'] = cal.apply(dE_for, axis=1)
cal['family'] = [family(r.base_form if r.kind == 'acid' else r.acid_form) for _, r in cal.iterrows()]
fits = {}
for kind in ['acid', 'base']:
    c = cal[(cal.kind == kind) & cal.dE.notna() & (cal.family != 'halide') & (cal.heavy <= CAL_MAX_HEAVY)].reset_index(drop=True)
    if len(c) < 5:
        print(f'{kind}: only {len(c)} finished pairs -- run more of Step 6 first.'); continue
    x, y = c.dE.values, c.pKa.values
    slope, icpt = np.polyfit(x, y, 1)
    loo = np.array([np.polyval(np.polyfit(np.delete(x, i), np.delete(y, i), 1), x[i]) for i in range(len(x))])
    c['loo_pred'] = loo; c['loo_err'] = np.abs(loo - y)
    fits[kind] = (slope, icpt, c)
    print(f'\n{kind.upper()}: n={len(c)}  slope={slope:.3f} pKa/(kcal/mol)  r2={np.corrcoef(x, y)[0,1]**2:.3f}'
          f'  leave-one-out MAE={c.loo_err.mean():.2f}')
    print(c.groupby('family').loo_err.agg(['count', 'mean']).round(2).to_string())
    print('  worst 5:', ', '.join(f"{i} ({e:.1f})" for i, e in
                                  c.sort_values('loo_err', ascending=False)[['id', 'loo_err']].head(5).values))
cal.to_csv(f'{OUT}/calibration_set.csv', index=False)

## Step 7b — R² of the DFT pKa vs measured pKa (all calibration ions, no filter)

Uses every ion that has a measured pKa and a finished DFT energy, including halides, large molecules and strong acids. **R2_LOO** is the honest number: each ion is predicted by a line fitted without it. Only ions with a measured value can be scored; the DFT-only ions have nothing to compare to.

In [ ]:
# Step 7b -- R2 of the DFT pKa against MEASURED pKa, on ALL calibration ions (no halide / size / range filter)
# Run right after Step 7 (needs `cal` with a finished 'dE' column).
import numpy as np, pandas as pd

def r2(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

allc = cal[cal.dE.notna()].reset_index(drop=True)
print(f'Ions with a finished DFT energy: {len(allc)} of {len(cal)}')
rows, parts = [], []
for kind in ['acid', 'base']:
    c = allc[allc.kind == kind].reset_index(drop=True)
    if len(c) < 5:
        print(kind, 'too few'); continue
    x, y = c.dE.values, c.pKa.values
    fit = np.polyval(np.polyfit(x, y, 1), x)
    loo = np.array([np.polyval(np.polyfit(np.delete(x, i), np.delete(y, i), 1), x[i]) for i in range(len(x))])
    c['fit_pred'], c['loo_pred'] = fit, loo
    parts.append(c)
    rows.append(dict(group=kind, n=len(c), R2_fit=r2(y, fit), R2_LOO=r2(y, loo),
                     MAE_fit=np.abs(y - fit).mean(), MAE_LOO=np.abs(y - loo).mean()))
both = pd.concat(parts, ignore_index=True)
rows.append(dict(group='ALL (acids + bases)', n=len(both), R2_fit=r2(both.pKa, both.fit_pred),
                 R2_LOO=r2(both.pKa, both.loo_pred), MAE_fit=np.abs(both.pKa - both.fit_pred).mean(),
                 MAE_LOO=np.abs(both.pKa - both.loo_pred).mean()))
res = pd.DataFrame(rows).round(3)
print(res.to_string(index=False))
print('\nUse R2_LOO in the presentation (each ion predicted by a line that never saw it).')
both.to_csv(f'{OUT}/DFT_vs_measured_all_ions.csv', index=False)
res.to_csv(f'{OUT}/DFT_R2_summary.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(fits), figsize=(6 * len(fits), 5), squeeze=False)
for ax, (kind, (s, b, c)) in zip(axes[0], fits.items()):
    for fam, g in c.groupby('family'):
        ax.scatter(g.dE, g.pKa, label=fam, s=35)
    xs = np.linspace(c.dE.min(), c.dE.max(), 50); ax.plot(xs, s * xs + b, 'k--', lw=1)
    ax.set_xlabel('ΔE (kcal/mol)'); ax.set_ylabel('experimental pKa'); ax.set_title(f'{kind}: slope {s:.3f}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/calibration.png', dpi=150); plt.show()

**Decision gate before spending hours on Step 8:** if the leave-one-out error is about 1 pKa unit or better —
and the sulfonate/sulfate and carboxylate families look sensible — go ahead. If a family is poor, you can
still run Step 8 but mark that family's predictions as low confidence.

## Step 8 — Run the unknown ions

Every ion without a literature value that has a valid one-proton pair. Resumable — expect several sessions.

In [ ]:
unk = []
for _, r in ions[ions['status'] != 'Source-backed value'].iterrows():
    acid, base, kind = make_pair(r['smiles'], r['ion_type'])
    unk.append(dict(master_id=r['master_id'], name=r['name'], ion_type=r['ion_type'], kind=kind,
                    acid_form=acid, base_form=base, sheet_status=r['status']))
unk = pd.DataFrame(unk)
unk['iupac_pKa'] = unk.master_id.map(IUPAC_VALUES)       # measured values need no DFT
runnable = unk[unk.acid_form.notna() & unk.base_form.notna() & unk.iupac_pKa.isna()]
print(f'{len(runnable)} of {len(unk)} ions have a one-proton pair; the rest are permanent cations or '
      f'non-protonatable anions (BF4-, SbF6-, ...).')
run_species(list(runnable.acid_form) + list(runnable.base_form))

## Step 9 — Predict and export

In [ ]:
d = load_cache()
unk['dE'] = unk.apply(lambda r: dE_for(r) if r.acid_form and r.base_form else np.nan, axis=1)
unk['family'] = [family(r.base_form if r.kind == 'acid' else r.acid_form) if r.acid_form and r.base_form else None
                 for _, r in unk.iterrows()]
def predict(r):
    if not pd.isna(r.iupac_pKa):
        return pd.Series([r.iupac_pKa, 'measured value (IUPAC dataset)'])
    if pd.isna(r.dE) or r.kind not in fits:
        return pd.Series([np.nan, 'not computed / no pair' if pd.isna(r.dE) else 'no calibration for this kind'])
    s, b, c = fits[r.kind]
    p = s * r.dE + b
    lo, hi = c.dE.min(), c.dE.max()
    flag = '' if lo <= r.dE <= hi else 'outside calibration range'
    if r.family == 'halide':
        flag = 'halide: DFT unreliable, use literature value'
    elif Chem.MolFromSmiles(r.base_form).GetNumHeavyAtoms() > CAL_MAX_HEAVY:
        flag = (flag + '; ' if flag else '') + 'large molecule: lower confidence'
    if not (-15 <= p <= 25):                      # no real aqueous pKa lies outside this; treat as a failure
        return pd.Series([np.nan, f'implausible result ({p:.0f}) -- check this ion'])
    return pd.Series([round(p, 2), flag])
unk[['pKa_dft', 'flag']] = unk.apply(predict, axis=1)
fam_err = {k: v[2].groupby('family').loo_err.mean().to_dict() for k, v in fits.items()}
unk['family_LOO_error'] = [fam_err.get(k, {}).get(f, np.nan) for k, f in zip(unk.kind, unk.family)]
out = unk[['master_id', 'name', 'ion_type', 'kind', 'acid_form', 'base_form', 'family', 'dE',
           'pKa_dft', 'family_LOO_error', 'flag', 'sheet_status']]
out.to_csv(f'{OUT}/IL_pKa_DFT_SMD.csv', index=False)
print(out.flag.replace('', 'ok').value_counts().to_string())
print(f'\nSaved: {OUT}/IL_pKa_DFT_SMD.csv')

---
## Limitations — report these with the numbers

1. **Single conformer, force-field geometry, single-point DFT.** No DFT geometry optimisation, no thermal
   corrections. Chosen to fit a free T4; the calibration absorbs part of the resulting error, not all of it.
2. **Implicit solvent only.** SMD's own paper reports about 4 kcal/mol average error for ions — the reason
   calibration is mandatory. Small, concentrated ions (F⁻, Cl⁻) are the known weak spot.
3. **Calibration quality sets the ceiling.** Families with few calibration members (check the Step 7
   breakdown) get less reliable predictions.
4. **Multiply charged anions** use the last protonation step (e.g. SO₄²⁻/HSO₄⁻), which is the equilibrium that
   matters for the salt — but this is the protonation site chosen by a simple rule (first O⁻).
5. **16 anions have no protonatable site** (BF₄⁻, PF₆⁻, SbF₆⁻, GaCl₄⁻, I₃⁻, ...) — their conjugate acids are
   superacids. Not computed; treat them as always fully deprotonated in ΔpKa screening.

## Citations

- Marenich, A.V., Cramer, C.J. & Truhlar, D.G. (2009). Universal solvation model based on solute electron
  density (SMD). *J. Phys. Chem. B*, 113, 6378–6396.
- Klamt, A., Eckert, F., Diedenhofen, M. & Beck, M.E. (2003). First principles calculations of aqueous pKa values
  for organic and inorganic acids using COSMO-RS. *J. Phys. Chem. A*, 107, 9380–9386. (calibration approach)
- Li, R., Sun, Q., Zhang, X. & Chan, G.K.-L. (2024). Introducing GPU acceleration into the Python-based
  Simulations of Chemistry Framework. arXiv:2407.09700. (GPU4PySCF)
- Zheng, J.W. *et al.* IUPAC Digitized pKa Dataset — github.com/IUPAC/Dissociation-Constants (CC BY-NC 4.0).
  (measured values and extra calibration)
- Baltruschat, M. & Czodrowski, P. (2020). Machine learning meets pKa. *F1000Research*, 9, 113. (extra
  calibration molecules, CC BY 4.0)
